# T16 long-running M3-C Colab recovery

This is deliberately separate from `colab_throughput_smoke.ipynb`. It runs only a **TO BE PRE-REGISTERED** recipe and protects progress with asynchronous full-state snapshots. Do not fill the parameter block until the run name, commit, recipe, cadence, replay policy, and expected duration are registered.

The notebook mounts Drive, starts a clean Python 3.12 workload environment, selects `cu128` for compute capability 10.0 or newer, and resumes from the newest verified complete Drive snapshot for `RUN_NAME`. It never releases a runtime while work remains.

In [ ]:
# ========================= TO BE PRE-REGISTERED =========================
# Fill every placeholder from the approved registration. This notebook intentionally
# does not choose an M3-C recipe, seed, init checkpoint, duration, or cadence.
RUN_NAME = 'TO_BE_PRE_REGISTERED'
PINNED_COMMIT = 'TO_BE_PRE_REGISTERED'  # exact 40-hex Git commit
PRE_REGISTERED_RECIPE_ARGS = [
    # e.g. '--init', '/content/drive/...', ... (registered arguments only)
]
REQUESTED_TRANSITIONS = 'TO_BE_PRE_REGISTERED'  # positive integer target for this run
FULL_STATE_EVERY = 'TO_BE_PRE_REGISTERED'  # positive integer transitions
FULL_STATE_REPLAY = 'TO_BE_PRE_REGISTERED'  # exactly 'none' or 'full'
# ========================================================================

REPOSITORY = 'https://github.com/josenunezwork/snake-dqn.git'
DRIVE_ROOT = '/content/drive/MyDrive/t16_colab_long_runs'
LOCAL_WORK_ROOT = '/content/t16-m3c-long-run'
RELEASE_AFTER_SUCCESS = True
DURABLE_CONFIRM_TIMEOUT_SECONDS = 900

def require_preregistration():
    if RUN_NAME == 'TO_BE_PRE_REGISTERED' or PINNED_COMMIT == 'TO_BE_PRE_REGISTERED':
        raise RuntimeError('Set RUN_NAME and PINNED_COMMIT from a completed preregistration.')
    if len(PINNED_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in PINNED_COMMIT):
        raise RuntimeError('PINNED_COMMIT must be an exact lowercase 40-hex commit.')
    if not PRE_REGISTERED_RECIPE_ARGS:
        raise RuntimeError('The training recipe is TO BE PRE-REGISTERED; do not run an empty recipe.')
    if not isinstance(REQUESTED_TRANSITIONS, int) or REQUESTED_TRANSITIONS <= 0:
        raise RuntimeError('REQUESTED_TRANSITIONS must be a positive registered integer.')
    if not isinstance(FULL_STATE_EVERY, int) or FULL_STATE_EVERY <= 0:
        raise RuntimeError('FULL_STATE_EVERY must be a positive registered integer.')
    if FULL_STATE_REPLAY not in {'none', 'full'}:
        raise RuntimeError("FULL_STATE_REPLAY must be registered as 'none' or 'full'.")
    controlled = {'--transitions', '--out', '--full-state-every', '--full-state-replay',
                  '--ckpt-local-dir', '--ckpt-durable-dir', '--resume-full-state'}
    if controlled.intersection(PRE_REGISTERED_RECIPE_ARGS):
        raise RuntimeError('Use the recovery notebook parameters for controlled output/state flags.')


In [ ]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

require_preregistration()
drive.mount('/content/drive')

run_root = Path(DRIVE_ROOT) / RUN_NAME
output_dir = run_root / 'trainer_output'
durable_ckpt_dir = run_root / 'full_state'
local_ckpt_dir = Path(LOCAL_WORK_ROOT) / RUN_NAME / 'full_state'
checkout = Path('/content/t16-snake-dqn-long-run')
venv = Path('/content/t16-m3c-py312')
shutil.rmtree(checkout, ignore_errors=True)
shutil.rmtree(venv, ignore_errors=True)
subprocess.run(['git', 'init', str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'remote', 'add', 'origin', REPOSITORY], check=True)
subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', PINNED_COMMIT], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != PINNED_COMMIT:
    raise RuntimeError(f'pinned checkout mismatch: {actual_commit}')

# The Colab kernel may be Python 3.13; install and run the workload in a supported 3.12 venv.
subprocess.run(['uv', 'venv', '--seed', '--python', '3.12', str(venv)], check=True)
python = str(venv / 'bin' / 'python')
subprocess.run([python, '-m', 'pip', 'install', '-r', str(checkout / 'requirements-gpu.txt'),
                'numpy==1.26.4', 'numba==0.68.0', 'llvmlite==0.50.0'], check=True)
compute_capability = float(subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True
).strip().splitlines()[0])
cuda_index = 'https://download.pytorch.org/whl/cu128' if compute_capability >= 10.0 else 'https://download.pytorch.org/whl/cu126'
print(f'GPU compute capability {compute_capability}; torch index {cuda_index}')
subprocess.run([python, '-m', 'pip', 'install', '--index-url', cuda_index, 'torch==2.9.1'], check=True)
subprocess.run([python, '-c', 'import torch; assert torch.cuda.is_available(), torch.__version__'], check=True)

# Verify Drive snapshots before deciding whether to resume. The trainer itself selects
# the newest valid snapshot, so a corrupt latest manifest falls back safely.
discover = (
    'from pathlib import Path; from research.redesign_m3c_20261008.async_checkpoint '
    'import discover_snapshots; import json; '
    f"print(json.dumps(discover_snapshots(Path({str(durable_ckpt_dir)!r}), verify=True)))"
)
snapshot_result = subprocess.run([python, '-c', discover], cwd=checkout, text=True, capture_output=True)
if snapshot_result.returncode:
    # An empty directory is a first run. A non-empty but invalid directory is unsafe.
    if durable_ckpt_dir.exists() and any(durable_ckpt_dir.iterdir()):
        raise RuntimeError(f'Drive snapshots are invalid: {snapshot_result.stderr}')
    snapshots = []
else:
    import json
    snapshots = json.loads(snapshot_result.stdout)
    invalid_complete_name = durable_ckpt_dir.exists() and any(
        path.name.startswith('snapshot-') for path in durable_ckpt_dir.iterdir()
    )
    if not snapshots and invalid_complete_name:
        raise RuntimeError('A named Drive snapshot exists but none verify; recovery is required.')
resume_args = ['--resume-full-state', str(durable_ckpt_dir)] if snapshots else []
print('auto-resume:', snapshots[0].get('snapshot') if snapshots else 'no verified snapshot')


In [ ]:
# The recipe arguments are intentionally supplied only by the preregistration block above.
# Async snapshots stay off Drive's critical path: local serialization first, durable copy second.
command = [
    python, 'research/redesign_m3c_20261008/train_m3c.py', *PRE_REGISTERED_RECIPE_ARGS,
    '--transitions', str(REQUESTED_TRANSITIONS),
    '--out', str(output_dir),
    '--full-state-every', str(FULL_STATE_EVERY),
    '--full-state-replay', FULL_STATE_REPLAY,
    '--ckpt-local-dir', str(local_ckpt_dir),
    '--ckpt-durable-dir', str(durable_ckpt_dir),
    *resume_args,
]
print('+', ' '.join(command))
run_started_ns = time.time_ns()
completed = subprocess.run(command, cwd=checkout)
if completed.returncode:
    raise RuntimeError(f'train_m3c exited {completed.returncode}; leaving runtime assigned for recovery.')
summary_path = output_dir / 'summary.json'
if not summary_path.is_file() or summary_path.stat().st_mtime_ns < run_started_ns:
    raise RuntimeError('The completed command did not write a fresh summary; runtime remains assigned.')
summary = json.loads(summary_path.read_text())
if (summary.get('transitions', -1) < REQUESTED_TRANSITIONS or summary.get('halt') is not None
        or summary.get('stopped_for_time') or summary.get('full_state_complete') is not True):
    raise RuntimeError('Training did not reach the registered transition target cleanly; runtime remains assigned.')

# A clean trainer exit is insufficient: wait until a hash-verified durable snapshot exists.
deadline = time.monotonic() + DURABLE_CONFIRM_TIMEOUT_SECONDS
while True:
    confirmed = subprocess.run([python, '-c', discover], cwd=checkout, text=True, capture_output=True)
    if confirmed.returncode == 0:
        manifests = json.loads(confirmed.stdout)
        if (manifests and manifests[0].get('transitions', -1) >= summary['transitions']
                and manifests[0].get('transitions') >= REQUESTED_TRANSITIONS):
            print('durable final snapshot confirmed:', manifests[0].get('snapshot'))
            break
    if time.monotonic() >= deadline:
        raise TimeoutError('No verified durable snapshot after successful training; runtime remains assigned.')
    time.sleep(10)

# Only release after both completion and durable verification. No failure path unassigns the runtime.
if RELEASE_AFTER_SUCCESS:
    from google.colab import runtime
    runtime.unassign()
